In [1]:
!pip install -q torch

In [2]:
import os, io, zipfile, random, warnings, pickle
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 3070 Ti
VRAM: 8.6 GB


## Checkpoint Utilities
Models and dataset metadata are saved to `./checkpoints/`. Re-run any cell — if a checkpoint exists it will be loaded instead of retraining.

In [3]:
CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def ckpt_path(name):
    return os.path.join(CHECKPOINT_DIR, name)

def save_model(model, name, extra_config=None):
    """Save model state_dict + optional config dict."""
    payload = {'state_dict': model.state_dict()}
    if extra_config:
        payload['config'] = extra_config
    torch.save(payload, ckpt_path(f'{name}.pt'))
    print(f"  [Saved] {ckpt_path(f'{name}.pt')}")

def load_model(model_cls, name, config, device=DEVICE):
    """Load model from checkpoint if it exists; return (model, loaded:bool)."""
    p = ckpt_path(f'{name}.pt')
    if os.path.exists(p):
        # weights_only=False needed because we stored a dict with config
        ckpt = torch.load(p, map_location=device, weights_only=False)
        m = model_cls(**config).to(device)
        m.load_state_dict(ckpt['state_dict'])
        m.eval()
        print(f"  [Loaded from checkpoint] {p}")
        return m, True
    return None, False

def save_dataset_meta(meta_dict):
    p = ckpt_path('dataset_meta.pkl')
    with open(p, 'wb') as f:
        pickle.dump(meta_dict, f)
    print(f"  [Saved] {p}")

def load_dataset_meta():
    p = ckpt_path('dataset_meta.pkl')
    if os.path.exists(p):
        with open(p, 'rb') as f:
            meta = pickle.load(f)
        print(f"  [Loaded dataset metadata from checkpoint] {p}")
        return meta, True
    return {}, False

print("Checkpoint dir ready:", CHECKPOINT_DIR)

Checkpoint dir ready: checkpoints


In [4]:
def load_movielens_100k(cache_dir='ml-100k'):
    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
    if not os.path.exists(cache_dir):
        print("Downloading MovieLens 100K...")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        zipfile.ZipFile(io.BytesIO(r.content)).extractall('.')
    ratings = pd.read_csv(
        f'{cache_dir}/u.data', sep='\t',
        names=['user_id', 'item_id', 'rating', 'timestamp']
    )
    genre_cols = [
        'unknown', 'action', 'adventure', 'animation', 'childrens', 'comedy',
        'crime', 'documentary', 'drama', 'fantasy', 'film_noir', 'horror',
        'musical', 'mystery', 'romance', 'sci_fi', 'thriller', 'war', 'western'
    ]
    items_df = pd.read_csv(
        f'{cache_dir}/u.item', sep='|', encoding='latin-1', header=None,
        usecols=list(range(24)),
        names=['item_id', 'title', 'release_date', 'video_date', 'imdb_url'] + genre_cols
    )[['item_id', 'title'] + genre_cols]
    return ratings, items_df

ratings, items_df = load_movielens_100k()
print(f"Ratings: {len(ratings):,}  Users: {ratings.user_id.nunique()}  Items: {ratings.item_id.nunique()}")

Ratings: 100,000  Users: 943  Items: 1682


In [5]:
def build_implicit_dataset(ratings, pos_threshold=3, min_interactions=3):
    pos = ratings[ratings['rating'] >= pos_threshold].copy()
    user_ids = sorted(pos['user_id'].unique())
    item_ids = sorted(pos['item_id'].unique())
    user2idx = {u: i for i, u in enumerate(user_ids)}
    item2idx = {v: i for i, v in enumerate(item_ids)}
    pos['user'] = pos['user_id'].map(user2idx)
    pos['item'] = pos['item_id'].map(item2idx)
    n_users, n_items = len(user2idx), len(item2idx)
    pos = pos.groupby('user').filter(lambda g: len(g) >= min_interactions)
    pos = pos.sort_values(['user', 'timestamp'])
    train_rows, test_rows = [], []
    user_pos = defaultdict(set)
    user_history = defaultdict(list)
    for uid, grp in pos.groupby('user'):
        items_sorted = grp['item'].tolist()
        test_rows.append({'user': uid, 'item': items_sorted[-1]})
        for it in items_sorted[:-1]:
            train_rows.append({'user': uid, 'item': it, 'label': 1})
            user_pos[uid].add(it)
            user_history[uid].append(it)
    return (pd.DataFrame(train_rows), pd.DataFrame(test_rows),
            user_pos, user_history, n_users, n_items, item2idx, user2idx)

# ── Try loading pre-built metadata to skip re-processing ─────────────
meta, meta_loaded = load_dataset_meta()

if meta_loaded:
    train_df      = meta['train_df']
    test_df       = meta['test_df']
    user_pos      = meta['user_pos']
    user_history  = meta['user_history']
    n_users       = meta['n_users']
    n_items       = meta['n_items']
    item2idx      = meta['item2idx']
    user2idx      = meta['user2idx']
    EVAL_NEGS     = meta['EVAL_NEGS']
else:
    (train_df, test_df, user_pos, user_history,
     n_users, n_items, item2idx, user2idx) = build_implicit_dataset(ratings)

    rng = np.random.RandomState(SEED)
    EVAL_NEGS = {}
    for _, row in test_df.iterrows():
        u, pos_item = int(row['user']), int(row['item'])
        exclude = user_pos[u] | {pos_item}
        negs = []
        while len(negs) < 99:
            c = int(rng.randint(0, n_items))
            if c not in exclude:
                negs.append(c)
                exclude.add(c)
        EVAL_NEGS[u] = negs

    save_dataset_meta({
        'train_df': train_df, 'test_df': test_df,
        'user_pos': user_pos, 'user_history': user_history,
        'n_users': n_users, 'n_items': n_items,
        'item2idx': item2idx, 'user2idx': user2idx,
        'EVAL_NEGS': EVAL_NEGS,
    })

idx2item = {v: k for k, v in item2idx.items()}
item_titles = dict(zip(items_df['item_id'], items_df['title']))
print(f"Train: {len(train_df):,}  Test users: {len(test_df)}  n_users={n_users}  n_items={n_items}")
print(f"Pre-sampled {len(EVAL_NEGS)} x 99 evaluation negatives")

  [Saved] checkpoints\dataset_meta.pkl
Train: 81,577  Test users: 943  n_users=943  n_items=1574
Pre-sampled 943 x 99 evaluation negatives


In [6]:
class ImplicitDataset(Dataset):
    def __init__(self, train_df, user_pos, n_items, neg_ratio=6):
        self.user_pos = user_pos
        self.n_items = n_items
        self.neg_ratio = neg_ratio
        self.positives = list(zip(train_df['user'].tolist(), train_df['item'].tolist()))

    def __len__(self):
        return len(self.positives) * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.neg_ratio)
        within  = idx % (1 + self.neg_ratio)
        u, i = self.positives[pos_idx]
        if within == 0:
            return (torch.tensor(u, dtype=torch.long),
                    torch.tensor(i, dtype=torch.long),
                    torch.tensor(1.0, dtype=torch.float32))
        neg = random.randint(0, self.n_items - 1)
        while neg in self.user_pos[u]:
            neg = random.randint(0, self.n_items - 1)
        return (torch.tensor(u, dtype=torch.long),
                torch.tensor(neg, dtype=torch.long),
                torch.tensor(0.0, dtype=torch.float32))


def train_model(model, train_df, user_pos, n_items, n_epochs=50, batch_size=1024,
                lr=1e-3, weight_decay=1e-5, neg_ratio=6, device=DEVICE):
    dataset  = ImplicitDataset(train_df, user_pos, n_items, neg_ratio)
    # num_workers=0 avoids Windows multiprocessing issues in VSCode/Jupyter
    loader   = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=(device.type == 'cuda'))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)
    model.train()
    for epoch in range(1, n_epochs + 1):
        total_loss = 0.0
        for users, items, labels in loader:
            users, items, labels = users.to(device), items.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(users, items), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * len(users)
        scheduler.step()
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{n_epochs} | Loss: {total_loss / len(dataset):.5f} | LR: {scheduler.get_last_lr()[0]:.2e}")
    return model

In [7]:
def evaluate_ranking(model, test_df, eval_negs, Ks=(5, 10, 15, 20), device=DEVICE):
    model.eval()
    results = {f'NDCG@{k}': [] for k in Ks}
    results.update({f'HR@{k}': [] for k in Ks})
    with torch.no_grad():
        for _, row in test_df.iterrows():
            u, pos_item = int(row['user']), int(row['item'])
            candidates = [pos_item] + eval_negs[u]
            u_t = torch.tensor([u] * 100, dtype=torch.long, device=device)
            i_t = torch.tensor(candidates, dtype=torch.long, device=device)
            scores = torch.sigmoid(model(u_t, i_t)).cpu().numpy()
            rank = int((scores[1:] >= scores[0]).sum()) + 1
            for k in Ks:
                results[f'HR@{k}'].append(1 if rank <= k else 0)
                results[f'NDCG@{k}'].append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)
    return {key: float(np.mean(vals)) for key, vals in results.items()}

In [8]:
class GMF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=128):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.output   = nn.Linear(emb_dim, 1)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, users, items):
        return self.output(self.user_emb(users) * self.item_emb(items)).squeeze(-1)


class MLPModel(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=128, mlp_dims=(512, 256, 128, 64), dropout=0.1):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_dims:
            layers += [nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        self.mlp     = nn.Sequential(*layers)
        self.predict = nn.Linear(mlp_dims[-1], 1)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
        nn.init.xavier_uniform_(self.predict.weight)
        nn.init.zeros_(self.predict.bias)

    def forward(self, users, items):
        x = torch.cat([self.user_emb(users), self.item_emb(items)], dim=-1)
        return self.predict(self.mlp(x)).squeeze(-1)


class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=128, mlp_dims=(512, 256, 128, 64), dropout=0.1):
        super().__init__()
        self.user_emb_gmf = nn.Embedding(n_users, emb_dim)
        self.item_emb_gmf = nn.Embedding(n_items, emb_dim)
        self.user_emb_mlp = nn.Embedding(n_users, emb_dim)
        self.item_emb_mlp = nn.Embedding(n_items, emb_dim)
        mlp_layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_dims:
            mlp_layers += [nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        self.mlp     = nn.Sequential(*mlp_layers)
        self.predict = nn.Linear(emb_dim + mlp_dims[-1], 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [self.user_emb_gmf, self.item_emb_gmf, self.user_emb_mlp, self.item_emb_mlp]:
            nn.init.normal_(emb.weight, std=0.01)
        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
        nn.init.xavier_uniform_(self.predict.weight)
        nn.init.zeros_(self.predict.bias)

    def forward(self, users, items):
        gmf_out = self.user_emb_gmf(users) * self.item_emb_gmf(items)
        mlp_out = self.mlp(torch.cat([self.user_emb_mlp(users), self.item_emb_mlp(items)], dim=-1))
        return self.predict(torch.cat([gmf_out, mlp_out], dim=-1)).squeeze(-1)


def init_neumf_from_pretrained(neumf, gmf, mlp):
    neumf.user_emb_gmf.weight.data.copy_(gmf.user_emb.weight.data)
    neumf.item_emb_gmf.weight.data.copy_(gmf.item_emb.weight.data)
    neumf.user_emb_mlp.weight.data.copy_(mlp.user_emb.weight.data)
    neumf.item_emb_mlp.weight.data.copy_(mlp.item_emb.weight.data)
    src_linears = [m for m in mlp.mlp.modules()   if isinstance(m, nn.Linear)]
    dst_linears = [m for m in neumf.mlp.modules() if isinstance(m, nn.Linear)]
    for src, dst in zip(src_linears, dst_linears):
        dst.weight.data.copy_(src.weight.data)
        dst.bias.data.copy_(src.bias.data)
    alpha = 0.5
    neumf.predict.weight.data = torch.cat([
        alpha * gmf.output.weight.data,
        (1 - alpha) * mlp.predict.weight.data
    ], dim=1)
    neumf.predict.bias.data.zero_()
    return neumf

In [9]:
HP = dict(
    emb_dim        = 128,
    mlp_dims       = (512, 256, 128, 64),
    dropout        = 0.1,
    pretrain_epochs = 80,
    finetune_epochs = 60,
    batch_size     = 1024,
    lr             = 1e-3,
    finetune_lr    = 5e-4,
    weight_decay   = 1e-5,
    neg_ratio      = 6,
)

# Shared model config dict (passed to both training and checkpoint loading)
MODEL_CFG = dict(n_users=n_users, n_items=n_items, emb_dim=HP['emb_dim'],
                 mlp_dims=HP['mlp_dims'], dropout=HP['dropout'])
GMF_CFG   = dict(n_users=n_users, n_items=n_items, emb_dim=HP['emb_dim'])

print(HP)

{'emb_dim': 128, 'mlp_dims': (512, 256, 128, 64), 'dropout': 0.1, 'pretrain_epochs': 80, 'finetune_epochs': 60, 'batch_size': 1024, 'lr': 0.001, 'finetune_lr': 0.0005, 'weight_decay': 1e-05, 'neg_ratio': 6}


## Pre-training GMF & MLP
If `checkpoints/gmf.pt` and `checkpoints/mlp.pt` already exist, training is skipped and weights are loaded directly.

In [10]:
# ── GMF ──────────────────────────────────────────────────────────────────
gmf, gmf_loaded = load_model(GMF, 'gmf', GMF_CFG)
if not gmf_loaded:
    print("Pre-training GMF...")
    torch.manual_seed(SEED)
    gmf = GMF(**GMF_CFG).to(DEVICE)
    print(f"  Parameters: {sum(p.numel() for p in gmf.parameters()):,}")
    gmf = train_model(gmf, train_df, user_pos, n_items,
                      n_epochs=HP['pretrain_epochs'], batch_size=HP['batch_size'],
                      lr=HP['lr'], weight_decay=HP['weight_decay'],
                      neg_ratio=HP['neg_ratio'])
    save_model(gmf, 'gmf', extra_config=GMF_CFG)
else:
    print(f"  GMF parameters: {sum(p.numel() for p in gmf.parameters()):,}")

# ── MLP ──────────────────────────────────────────────────────────────────
mlp, mlp_loaded = load_model(MLPModel, 'mlp', MODEL_CFG)
if not mlp_loaded:
    print("\nPre-training MLP...")
    torch.manual_seed(SEED)
    mlp = MLPModel(**MODEL_CFG).to(DEVICE)
    print(f"  Parameters: {sum(p.numel() for p in mlp.parameters()):,}")
    mlp = train_model(mlp, train_df, user_pos, n_items,
                      n_epochs=HP['pretrain_epochs'], batch_size=HP['batch_size'],
                      lr=HP['lr'], weight_decay=HP['weight_decay'],
                      neg_ratio=HP['neg_ratio'])
    save_model(mlp, 'mlp', extra_config=MODEL_CFG)
else:
    print(f"  MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}")

print("\nGMF and MLP ready.")

Pre-training GMF...
  Parameters: 322,305
  Epoch   1/80 | Loss: 0.43531 | LR: 1.00e-03
  Epoch  10/80 | Loss: 0.22369 | LR: 9.62e-04
  Epoch  20/80 | Loss: 0.14347 | LR: 8.54e-04
  Epoch  30/80 | Loss: 0.07845 | LR: 6.92e-04
  Epoch  40/80 | Loss: 0.05062 | LR: 5.01e-04
  Epoch  50/80 | Loss: 0.03920 | LR: 3.09e-04
  Epoch  60/80 | Loss: 0.03474 | LR: 1.47e-04
  Epoch  70/80 | Loss: 0.03247 | LR: 3.90e-05
  Epoch  80/80 | Loss: 0.03215 | LR: 1.00e-06
  [Saved] checkpoints\gmf.pt

Pre-training MLP...
  Parameters: 628,225
  Epoch   1/80 | Loss: 0.30710 | LR: 1.00e-03
  Epoch  10/80 | Loss: 0.19862 | LR: 9.62e-04
  Epoch  20/80 | Loss: 0.15023 | LR: 8.54e-04
  Epoch  30/80 | Loss: 0.10886 | LR: 6.92e-04
  Epoch  40/80 | Loss: 0.07804 | LR: 5.01e-04
  Epoch  50/80 | Loss: 0.05575 | LR: 3.09e-04
  Epoch  60/80 | Loss: 0.04042 | LR: 1.47e-04
  Epoch  70/80 | Loss: 0.03150 | LR: 3.90e-05
  Epoch  80/80 | Loss: 0.02900 | LR: 1.00e-06
  [Saved] checkpoints\mlp.pt

GMF and MLP ready.


## NeuMF Fine-tuning
If `checkpoints/neumf.pt` exists the fine-tuned model is loaded directly. Otherwise it is initialised from the pretrained GMF/MLP weights and fine-tuned.

In [11]:
model, neumf_loaded = load_model(NeuMF, 'neumf', MODEL_CFG)

if not neumf_loaded:
    print("Initializing NeuMF from pre-trained GMF + MLP...")
    torch.manual_seed(SEED)
    model = NeuMF(**MODEL_CFG).to(DEVICE)
    model = init_neumf_from_pretrained(model, gmf, mlp)
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Free GPU memory from pre-trained models before fine-tuning
    del gmf, mlp
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    print("\nFine-tuning NeuMF...")
    model = train_model(model, train_df, user_pos, n_items,
                        n_epochs=HP['finetune_epochs'], batch_size=HP['batch_size'],
                        lr=HP['finetune_lr'], weight_decay=HP['weight_decay'],
                        neg_ratio=HP['neg_ratio'])
    save_model(model, 'neumf', extra_config=MODEL_CFG)
else:
    # If we loaded NeuMF from disk, gmf/mlp are no longer needed
    try:
        del gmf, mlp
    except NameError:
        pass
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    print(f"  NeuMF parameters: {sum(p.numel() for p in model.parameters()):,}")

print("\nNeuMF ready.")

Initializing NeuMF from pre-trained GMF + MLP...
  Parameters: 950,529

Fine-tuning NeuMF...
  Epoch   1/60 | Loss: 0.05075 | LR: 5.00e-04
  Epoch  10/60 | Loss: 0.02469 | LR: 4.67e-04
  Epoch  20/60 | Loss: 0.02062 | LR: 3.75e-04
  Epoch  30/60 | Loss: 0.01691 | LR: 2.51e-04
  Epoch  40/60 | Loss: 0.01325 | LR: 1.26e-04
  Epoch  50/60 | Loss: 0.01068 | LR: 3.44e-05
  Epoch  60/60 | Loss: 0.01014 | LR: 1.00e-06
  [Saved] checkpoints\neumf.pt

NeuMF ready.


In [12]:
results = evaluate_ranking(model, test_df, EVAL_NEGS)
print("NeuMF (Pre-trained Init) Results")
print("=" * 40)
for k in [5, 10, 15, 20]:
    print(f"  NDCG@{k:2d} = {results[f'NDCG@{k}']:.4f}    HR@{k:2d} = {results[f'HR@{k}']:.4f}")

NeuMF (Pre-trained Init) Results
  NDCG@ 5 = 0.3042    HR@ 5 = 0.4528
  NDCG@10 = 0.3526    HR@10 = 0.6034
  NDCG@15 = 0.3736    HR@15 = 0.6829
  NDCG@20 = 0.3885    HR@20 = 0.7455


In [13]:
@torch.no_grad()
def top_k_for_user(model, u, k=10):
    model.eval()
    unseen = [i for i in range(n_items) if i not in user_pos[u]]
    scores = []
    for s in range(0, len(unseen), 4096):
        batch = unseen[s:s + 4096]
        u_t = torch.tensor([u] * len(batch), dtype=torch.long, device=DEVICE)
        i_t = torch.tensor(batch,            dtype=torch.long, device=DEVICE)
        scores.append(torch.sigmoid(model(u_t, i_t)).cpu())
    scores  = torch.cat(scores).numpy()
    top_idx = np.argsort(scores)[::-1][:k]
    rows = []
    for rank, ci in enumerate(top_idx, 1):
        item_orig = idx2item.get(unseen[ci], -1)
        rows.append({'Rank': rank,
                     'Title': item_titles.get(item_orig, 'Unknown'),
                     'Score': round(float(scores[ci]), 4)})
    return pd.DataFrame(rows)

for K in [5, 10]:
    print(f"\nTop-{K} recommendations for user 0:")
    print(top_k_for_user(model, 0, k=K).to_string(index=False))


Top-5 recommendations for user 0:
 Rank                                                                       Title  Score
    1 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963) 0.1485
    2                                                       Close Shave, A (1995) 0.0787
    3                                                                 Heat (1995) 0.0736
    4                                                            True Lies (1994) 0.0508
    5                                           Jackie Chan's First Strike (1996) 0.0474

Top-10 recommendations for user 0:
 Rank                                                                       Title  Score
    1 Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963) 0.1485
    2                                                       Close Shave, A (1995) 0.0787
    3                                                                 Heat (1995) 0.0736
    4                                  